# 12 - App Data Hub

## Purpose of this notebook

In this notebook, I centralize the final strategy data that the dashboard app will use.

The earlier notebooks produced many intermediate research outputs, but the app should not depend on all of those scattered files directly. Instead, this notebook creates a clean app-facing data layer.

## What this notebook does

This notebook:
- loads the final selected strategy outputs
- standardizes strategy names
- combines return series into one file
- combines metrics into one file
- creates a clean registry table for the app
- saves weight histories for the strategies the app should support

## Why this matters

This makes the dashboard easier to maintain and makes it much easier to add or remove strategies later.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
APP_DIR = PROCESSED_DIR / "app_data"

APP_DIR.mkdir(parents=True, exist_ok=True)

print("Processed dir:", PROCESSED_DIR.resolve())
print("App data dir:", APP_DIR.resolve())

Processed dir: /Users/nicholasturangan/Desktop/quant/quant-portfolio-project/data/processed
App data dir: /Users/nicholasturangan/Desktop/quant/quant-portfolio-project/data/processed/app_data


In [3]:
def safe_read_csv(path, index_col=0, parse_dates=True):
    if path.exists():
        return pd.read_csv(path, index_col=index_col, parse_dates=parse_dates)
    return None


def compute_metrics(return_series):
    rs = return_series.dropna()
    if len(rs) == 0:
        return {
            "CAGR": np.nan,
            "Annual Vol": np.nan,
            "Sharpe": np.nan,
            "Max Drawdown": np.nan,
            "Calmar": np.nan,
        }

    cumulative = (1 + rs).cumprod()
    total_months = len(rs)

    cagr = cumulative.iloc[-1] ** (12 / total_months) - 1
    ann_vol = rs.std() * np.sqrt(12)
    sharpe = cagr / ann_vol if ann_vol != 0 else np.nan

    running_max = cumulative.cummax()
    drawdown = cumulative / running_max - 1
    max_dd = drawdown.min()

    calmar = cagr / abs(max_dd) if max_dd != 0 else np.nan

    return {
        "CAGR": cagr,
        "Annual Vol": ann_vol,
        "Sharpe": sharpe,
        "Max Drawdown": max_dd,
        "Calmar": calmar,
    }

In [4]:
final_comparison_returns = safe_read_csv(PROCESSED_DIR / "final_comparison_returns.csv")
final_comparison_metrics = safe_read_csv(PROCESSED_DIR / "final_comparison_metrics.csv")
final_strategy_summary = safe_read_csv(PROCESSED_DIR / "final_strategy_summary_table.csv")

risk_overlay_returns = safe_read_csv(PROCESSED_DIR / "risk_overlay_returns.csv")
risk_overlay_metrics = safe_read_csv(PROCESSED_DIR / "risk_overlay_metrics.csv")

portfolio_construction_overlay_returns = safe_read_csv(PROCESSED_DIR / "portfolio_construction_upgrade_overlay_returns.csv")
portfolio_construction_overlay_metrics = safe_read_csv(PROCESSED_DIR / "portfolio_construction_upgrade_overlay_metrics.csv")

print("Loaded:")
print("final_comparison_returns:", None if final_comparison_returns is None else final_comparison_returns.shape)
print("risk_overlay_returns:", None if risk_overlay_returns is None else risk_overlay_returns.shape)
print("portfolio_construction_overlay_returns:", None if portfolio_construction_overlay_returns is None else portfolio_construction_overlay_returns.shape)

Loaded:
final_comparison_returns: (62, 10)
risk_overlay_returns: (242, 12)
portfolio_construction_overlay_returns: (242, 3)


/var/folders/sx/02p4fzbd32gghvv7j9kwgb5m0000gn/T/ipykernel_9664/2241236284.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.read_csv(path, index_col=index_col, parse_dates=parse_dates)
/var/folders/sx/02p4fzbd32gghvv7j9kwgb5m0000gn/T/ipykernel_9664/2241236284.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.read_csv(path, index_col=index_col, parse_dates=parse_dates)
/var/folders/sx/02p4fzbd32gghvv7j9kwgb5m0000gn/T/ipykernel_9664/2241236284.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.read_csv(path, index_col=index_col, parse_dates

## Strategy selection

These are the strategies I actually want the app to support.

I keep:
- core base strategies
- best overlay upgrades
- final math-upgraded winner
- public benchmarks

In [5]:
strategy_specs = [
    {
        "display_name": "Shared 180d",
        "column_name": "Shared_180d",
        "source": "final_comparison_returns",
        "category": "base",
        "description": "Shared 180-day momentum strategy with inverse-vol weighting.",
        "is_default": False,
    },
    {
        "display_name": "Shared 180d + VolTarget + Sleeve",
        "column_name": "Shared_180d_VolTarget_Sleeve",
        "source": "risk_overlay_returns",
        "category": "overlay",
        "description": "Shared 180-day strategy with portfolio-level vol targeting and defensive sleeve.",
        "is_default": False,
    },
    {
        "display_name": "Shared 252d",
        "column_name": "Shared_252d",
        "source": "final_comparison_returns",
        "category": "base",
        "description": "Shared 252-day momentum strategy with inverse-vol weighting.",
        "is_default": False,
    },
    {
        "display_name": "Shared 252d + VolTarget + Sleeve",
        "column_name": "Shared_252d_VolTarget_Sleeve",
        "source": "risk_overlay_returns",
        "category": "overlay",
        "description": "Shared 252-day strategy with portfolio-level vol targeting and defensive sleeve.",
        "is_default": False,
    },
    {
        "display_name": "Shared 252d + MinVar + VolTarget + Sleeve",
        "column_name": "Shared_252d_MinVar_VolTarget_Sleeve",
        "source": "portfolio_construction_overlay_returns",
        "category": "final_winner",
        "description": "Final winning strategy: shared 252-day signal with minimum variance weighting, vol targeting, and defensive sleeve.",
        "is_default": True,
    },
    {
        "display_name": "Best Static ETF-Level",
        "column_name": "Best_Static_ETF_Level",
        "source": "final_comparison_returns",
        "category": "advanced",
        "description": "Static ETF-specific lookback strategy.",
        "is_default": False,
    },
    {
        "display_name": "Best Static ETF-Level + VolTarget + Sleeve",
        "column_name": "Best_Static_ETF_Level_VolTarget_Sleeve",
        "source": "risk_overlay_returns",
        "category": "advanced_overlay",
        "description": "ETF-level strategy with portfolio-level vol targeting and defensive sleeve.",
        "is_default": False,
    },
]

benchmark_specs = [
    {
        "display_name": "SPY Buy & Hold",
        "column_name": "SPY_BuyHold",
        "source": "final_comparison_returns",
        "category": "benchmark",
        "description": "SPY benchmark.",
        "is_default": False,
    },
    {
        "display_name": "60/40 Portfolio",
        "column_name": "Portfolio_60_40",
        "source": "final_comparison_returns",
        "category": "benchmark",
        "description": "60/40 benchmark.",
        "is_default": False,
    },
    {
        "display_name": "DBMF",
        "column_name": "DBMF",
        "source": "final_comparison_returns",
        "category": "benchmark",
        "description": "Managed futures benchmark.",
        "is_default": False,
    },
    {
        "display_name": "KMLM",
        "column_name": "KMLM",
        "source": "final_comparison_returns",
        "category": "benchmark",
        "description": "Managed futures / trend benchmark.",
        "is_default": False,
    },
    {
        "display_name": "RPAR",
        "column_name": "RPAR",
        "source": "final_comparison_returns",
        "category": "benchmark",
        "description": "Risk parity benchmark.",
        "is_default": False,
    },
]

all_specs = strategy_specs + benchmark_specs
registry_df = pd.DataFrame(all_specs)
registry_df

,display_name,column_name,source,category,description,is_default
0,Shared 180d,Shared_180d,final_comparison_returns,base,Shared 180-day momentum strategy with inverse-...,False
1,Shared 180d + VolTarget + Sleeve,Shared_180d_VolTarget_Sleeve,risk_overlay_returns,overlay,Shared 180-day strategy with portfolio-level v...,False
2,Shared 252d,Shared_252d,final_comparison_returns,base,Shared 252-day momentum strategy with inverse-...,False
3,Shared 252d + VolTarget + Sleeve,Shared_252d_VolTarget_Sleeve,risk_overlay_returns,overlay,Shared 252-day strategy with portfolio-level v...,False
4,Shared 252d + MinVar + VolTarget + Sleeve,Shared_252d_MinVar_VolTarget_Sleeve,portfolio_construction_overlay_returns,final_winner,Final winning strategy: shared 252-day signal ...,True
5,Best Static ETF-Level,Best_Static_ETF_Level,final_comparison_returns,advanced,Static ETF-specific lookback strategy.,False
6,Best Static ETF-Level + VolTarget + Sleeve,Best_Static_ETF_Level_VolTarget_Sleeve,risk_overlay_returns,advanced_overlay,ETF-level strategy with portfolio-level vol ta...,False
7,SPY Buy & Hold,SPY_BuyHold,final_comparison_returns,benchmark,SPY benchmark.,False
8,60/40 Portfolio,Portfolio_60_40,final_comparison_returns,benchmark,60/40 benchmark.,False
9,DBMF,DBMF,final_comparison_returns,benchmark,Managed futures benchmark.,False


In [6]:
source_map = {
    "final_comparison_returns": final_comparison_returns,
    "risk_overlay_returns": risk_overlay_returns,
    "portfolio_construction_overlay_returns": portfolio_construction_overlay_returns,
}

series_dict = {}

for _, row in registry_df.iterrows():
    source_df = source_map[row["source"]]
    col = row["column_name"]

    if source_df is None:
        print(f"Missing source dataframe for {col}")
        continue

    if col not in source_df.columns:
        print(f"Missing column: {col}")
        continue

    series_dict[col] = source_df[col]

app_strategy_returns = pd.DataFrame(series_dict).dropna(how="all")
app_strategy_returns.head()

,Shared_180d,Shared_180d_VolTarget_Sleeve,Shared_252d,Shared_252d_VolTarget_Sleeve,Shared_252d_MinVar_VolTarget_Sleeve,Best_Static_ETF_Level,Best_Static_ETF_Level_VolTarget_Sleeve,SPY_BuyHold,Portfolio_60_40,DBMF,KMLM,RPAR
2006-01-31,NaN,0.046116,NaN,0.046116,0.046116,NaN,0.046116,NaN,NaN,NaN,NaN,NaN
2006-02-28,NaN,-0.024799,NaN,-0.024799,-0.024806,NaN,-0.024799,NaN,NaN,NaN,NaN,NaN
2006-03-31,NaN,0.029098,NaN,0.029098,0.036205,NaN,0.029098,NaN,NaN,NaN,NaN,NaN
2006-04-28,NaN,0.095202,NaN,0.095202,0.120310,NaN,0.095202,NaN,NaN,NaN,NaN,NaN
2006-05-31,NaN,-0.063030,NaN,-0.063030,-0.013212,NaN,-0.063030,NaN,NaN,NaN,NaN,NaN


In [7]:
app_strategy_metrics = pd.DataFrame(
    {col: compute_metrics(app_strategy_returns[col]) for col in app_strategy_returns.columns}
).T

app_strategy_metrics

,CAGR,Annual Vol,Sharpe,Max Drawdown,Calmar
Shared_180d,0.123895,0.120031,1.032194,-0.238701,0.519038
Shared_180d_VolTarget_Sleeve,0.117661,0.108521,1.084227,-0.203339,0.578643
Shared_252d,0.151873,0.131968,1.150828,-0.225921,0.672238
Shared_252d_VolTarget_Sleeve,0.128758,0.112870,1.140756,-0.201629,0.638589
Shared_252d_MinVar_VolTarget_Sleeve,0.134767,0.113790,1.184351,-0.206275,0.653335
Best_Static_ETF_Level,0.122227,0.126248,0.968148,-0.227819,0.536510
Best_Static_ETF_Level_VolTarget_Sleeve,0.118921,0.106070,1.121156,-0.203431,0.584578
SPY_BuyHold,0.139733,0.149153,0.936839,-0.239272,0.583992
Portfolio_60_40,0.078093,0.108559,0.719360,-0.205075,0.380804
DBMF,0.106964,0.120924,0.884556,-0.172985,0.618344


In [8]:
app_strategy_returns.to_csv(APP_DIR / "app_strategy_returns.csv")
app_strategy_metrics.to_csv(APP_DIR / "app_strategy_metrics.csv")
registry_df.to_csv(APP_DIR / "app_strategy_registry.csv", index=False)

print("Saved:")
print(APP_DIR / "app_strategy_returns.csv")
print(APP_DIR / "app_strategy_metrics.csv")
print(APP_DIR / "app_strategy_registry.csv")

Saved:
../data/processed/app_data/app_strategy_returns.csv
../data/processed/app_data/app_strategy_metrics.csv
../data/processed/app_data/app_strategy_registry.csv


## App weight files

Here I create clean app-facing weight filenames so the dashboard does not need to know the original notebook-specific file names.

In [9]:
weight_file_map = {
    "Shared_180d": PROCESSED_DIR / "shared_180d_target_weights.csv",
    "Shared_180d_VolTarget_Sleeve": PROCESSED_DIR / "shared_180d_vol_target_sleeve_weights.csv",
    "Shared_252d": PROCESSED_DIR / "shared_252d_target_weights.csv",
    "Shared_252d_VolTarget_Sleeve": PROCESSED_DIR / "shared_252d_vol_target_sleeve_weights.csv",
    "Shared_252d_MinVar_VolTarget_Sleeve": PROCESSED_DIR / "shared_252d_minvar_vol_target_sleeve_weights.csv",
    "Best_Static_ETF_Level": PROCESSED_DIR / "best_etf_level_weights.csv",
    "Best_Static_ETF_Level_VolTarget_Sleeve": PROCESSED_DIR / "best_static_etf_level_vol_target_sleeve_weights.csv",
}

saved_weight_files = []

for strategy_col, source_path in weight_file_map.items():
    if source_path.exists():
        df = pd.read_csv(source_path, index_col=0, parse_dates=True)
        out_path = APP_DIR / f"app_weights_{strategy_col.lower()}.csv"
        df.to_csv(out_path)
        saved_weight_files.append(out_path.name)
    else:
        print(f"Missing weight file for {strategy_col}: {source_path.name}")

saved_weight_files

['app_weights_shared_180d.csv',
 'app_weights_shared_180d_voltarget_sleeve.csv',
 'app_weights_shared_252d.csv',
 'app_weights_shared_252d_voltarget_sleeve.csv',
 'app_weights_shared_252d_minvar_voltarget_sleeve.csv',
 'app_weights_best_static_etf_level.csv',
 'app_weights_best_static_etf_level_voltarget_sleeve.csv']

In [10]:
app_config = pd.DataFrame([
    {
        "default_strategy": "Shared_252d_MinVar_VolTarget_Sleeve",
        "default_display_name": "Shared 252d + MinVar + VolTarget + Sleeve",
        "recommended_strategy": "Shared_252d_MinVar_VolTarget_Sleeve",
        "recommended_display_name": "Shared 252d + MinVar + VolTarget + Sleeve",
        "recommended_reason": "Best overall balance of CAGR, Sharpe, and drawdown after all refinements.",
    }
])

app_config.to_csv(APP_DIR / "app_config.csv", index=False)
app_config

,default_strategy,default_display_name,recommended_strategy,recommended_display_name,recommended_reason
0,Shared_252d_MinVar_VolTarget_Sleeve,Shared 252d + MinVar + VolTarget + Sleeve,Shared_252d_MinVar_VolTarget_Sleeve,Shared 252d + MinVar + VolTarget + Sleeve,"Best overall balance of CAGR, Sharpe, and draw..."


In [11]:
print("App data files created:")
for p in sorted(APP_DIR.glob("*")):
    print("-", p.name)

App data files created:
- app_config.csv
- app_strategy_metrics.csv
- app_strategy_registry.csv
- app_strategy_returns.csv
- app_weights_best_static_etf_level.csv
- app_weights_best_static_etf_level_voltarget_sleeve.csv
- app_weights_shared_180d.csv
- app_weights_shared_180d_voltarget_sleeve.csv
- app_weights_shared_252d.csv
- app_weights_shared_252d_minvar_voltarget_sleeve.csv
- app_weights_shared_252d_voltarget_sleeve.csv
